# Day 3 · Deliver & Govern — Lab 3
# Rollout, RAID & Delivery Tabletop · Portfolio Governance & Reuse
# + Use Cases: Partial Payments · ERP Integration (incl. D365) · Reporting

Standard-library only, fully runnable top to bottom.

**Contents**
1. RAID Log (Risks, Assumptions, Issues, Dependencies) — structured tracker
2. Rollout Plan — phased rollout with go/no-go gates (canary -> partial -> full)
3. Delivery Tabletop Exercise — scripted incident simulation you can run as a team exercise
4. Portfolio Governance — reuse catalog + duplicate-build detector across teams
5. Use Case: Partial Payments — variance allocation + adjustment suggestions
6. Use Case: ERP Integration (D365-style) — journal entry preparation + posting simulation
7. Use Case: Reporting — generate a governance/ops report from everything above


In [1]:
import json, time, uuid, random
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional
from datetime import date, datetime, timedelta

random.seed(3)
print("Ready.")


Ready.


## 1. RAID Log (Risks, Assumptions, Issues, Dependencies)

The standard delivery-governance artifact. Distinct from the Day-3-Lab-1 risk register:
a RAID log spans the whole delivery, not just agent-specific risk categories.

In [2]:
RAID_TYPES = ["Risk", "Assumption", "Issue", "Dependency"]
RAID_PLURALS = {"Risk": "RISKS", "Assumption": "ASSUMPTIONS", "Issue": "ISSUES", "Dependency": "DEPENDENCIES"}


@dataclass
class RAIDItem:
    item_id: str
    item_type: str        # one of RAID_TYPES
    description: str
    owner: str
    status: str = "open"   # open | in_progress | resolved | accepted
    due_date: Optional[str] = None
    impact: str = "medium"  # low | medium | high


class RAIDLog:
    def __init__(self):
        self.items: List[RAIDItem] = []

    def add(self, item: RAIDItem):
        assert item.item_type in RAID_TYPES
        self.items.append(item)

    def by_type(self, item_type: str) -> List[RAIDItem]:
        return [i for i in self.items if i.item_type == item_type]

    def open_high_impact(self) -> List[RAIDItem]:
        return [i for i in self.items if i.status == "open" and i.impact == "high"]

    def report(self) -> str:
        lines = []
        for t in RAID_TYPES:
            entries = self.by_type(t)
            lines.append(f"\n{RAID_PLURALS[t]} ({len(entries)})")
            lines.append("-" * 50)
            for e in entries:
                lines.append(f"  [{e.status:11s}][{e.impact:6s}] {e.item_id}: {e.description} (owner: {e.owner})")
        return "\n".join(lines)


raid = RAIDLog()
raid.add(RAIDItem("RA-01", "Risk", "D365 sandbox environment may not mirror prod field mappings",
                   "Integration Lead", impact="high", due_date="2026-09-01"))
raid.add(RAIDItem("RA-02", "Assumption", "Remittance email volume stays under 5k/day through rollout",
                   "Product Owner", impact="medium"))
raid.add(RAIDItem("RA-03", "Issue", "Partial-payment variance threshold not yet agreed with Finance",
                   "Finance Lead", impact="high", status="open", due_date="2026-08-25"))
raid.add(RAIDItem("RA-04", "Dependency", "ERP posting requires updated D365 API credentials from IT",
                   "IT Ops", impact="high", status="in_progress"))
raid.add(RAIDItem("RA-05", "Risk", "Reviewer capacity may be insufficient during month-end close spike",
                   "Ops Manager", impact="medium"))

print(raid.report())
print("\n\nOPEN HIGH-IMPACT ITEMS (rollout blockers to watch):")
for i in raid.open_high_impact():
    print(f"  - {i.item_id}: {i.description}")



RISKS (2)
--------------------------------------------------
  [open       ][high  ] RA-01: D365 sandbox environment may not mirror prod field mappings (owner: Integration Lead)
  [open       ][medium] RA-05: Reviewer capacity may be insufficient during month-end close spike (owner: Ops Manager)

ASSUMPTIONS (1)
--------------------------------------------------
  [open       ][medium] RA-02: Remittance email volume stays under 5k/day through rollout (owner: Product Owner)

ISSUES (1)
--------------------------------------------------
  [open       ][high  ] RA-03: Partial-payment variance threshold not yet agreed with Finance (owner: Finance Lead)

DEPENDENCIES (1)
--------------------------------------------------
  [in_progress][high  ] RA-04: ERP posting requires updated D365 API credentials from IT (owner: IT Ops)


OPEN HIGH-IMPACT ITEMS (rollout blockers to watch):
  - RA-01: D365 sandbox environment may not mirror prod field mappings
  - RA-03: Partial-payment variance thresho

## 2. Rollout Plan — phased rollout with go/no-go gates

A canary -> partial -> full rollout plan, each phase with explicit **go/no-go criteria**
evaluated against live metrics before advancing.

In [3]:
@dataclass
class RolloutPhase:
    name: str
    traffic_pct: int
    go_criteria: Dict[str, float]     # metric_name -> minimum acceptable value (or max, see notes)


ROLLOUT_PLAN = [
    RolloutPhase("Canary", 5, {"accuracy": 0.95, "max_error_rate": 0.02}),
    RolloutPhase("Partial", 25, {"accuracy": 0.96, "max_error_rate": 0.015}),
    RolloutPhase("Full", 100, {"accuracy": 0.97, "max_error_rate": 0.01}),
]


def evaluate_gate(phase: RolloutPhase, observed: Dict[str, float]) -> Dict[str, Any]:
    failures = []
    for metric, threshold in phase.go_criteria.items():
        observed_value = observed.get(metric)
        if observed_value is None:
            failures.append(f"missing metric: {metric}")
            continue
        if metric.startswith("max_"):
            if observed_value > threshold:
                failures.append(f"{metric}={observed_value} exceeds max {threshold}")
        else:
            if observed_value < threshold:
                failures.append(f"{metric}={observed_value} below min {threshold}")
    return {"phase": phase.name, "go": len(failures) == 0, "failures": failures}


# Simulated observed metrics per phase
observed_metrics = {
    "Canary": {"accuracy": 0.965, "max_error_rate": 0.012},
    "Partial": {"accuracy": 0.958, "max_error_rate": 0.018},   # will fail max_error_rate gate
    "Full": {"accuracy": 0.978, "max_error_rate": 0.008},
}

for phase in ROLLOUT_PLAN:
    outcome = evaluate_gate(phase, observed_metrics[phase.name])
    verdict = "GO" if outcome["go"] else "NO-GO"
    print(f"Phase: {phase.name:8s} (traffic {phase.traffic_pct}%)  -> {verdict}")
    if not outcome["go"]:
        for f in outcome["failures"]:
            print(f"    blocked by: {f}")


Phase: Canary   (traffic 5%)  -> GO
Phase: Partial  (traffic 25%)  -> NO-GO
    blocked by: accuracy=0.958 below min 0.96
    blocked by: max_error_rate=0.018 exceeds max 0.015
Phase: Full     (traffic 100%)  -> GO


## 3. Delivery Tabletop Exercise — scripted incident simulation

A tabletop exercise walks the delivery team through a **simulated incident** to rehearse
the response before it happens for real. This is a runnable scripted version: it presents
an injected scenario, the expected response steps, and scores how a team's chosen actions
compare to the playbook.

In [4]:
@dataclass
class TabletopScenario:
    scenario_id: str
    injection: str                 # what happened
    expected_actions: List[str]    # ordered playbook steps


TABLETOP_SCENARIOS = [
    TabletopScenario(
        "TT-01",
        "Smart-matching agent auto-approved 40 remittances against the WRONG invoices "
        "overnight due to a currency-symbol parsing bug.",
        [
            "Trigger circuit breaker / pause auto-approval for the affected agent",
            "Pull audit trail for all auto-approved matches in the affected window",
            "Notify Finance + affected customers' account owners",
            "Roll back / reverse the incorrect ERP postings",
            "Root-cause the parsing bug and add a regression test to the golden set",
            "Post-incident review + update the risk register",
        ],
    ),
    TabletopScenario(
        "TT-02",
        "D365 API credentials expired mid-day, causing journal-entry postings to silently queue "
        "instead of posting.",
        [
            "Alert fires from failed-posting queue depth metric",
            "On-call rotates credentials / escalates to IT Ops dependency owner",
            "Confirm no duplicate postings once queue drains",
            "Reconcile queued vs. posted entries against source remittances",
            "Add expiry-monitoring alert for API credentials to prevent recurrence",
        ],
    ),
]


def score_tabletop_response(scenario: TabletopScenario, team_actions: List[str]) -> Dict[str, Any]:
    expected_set = set(a.lower() for a in scenario.expected_actions)
    team_set = set(a.lower() for a in team_actions)
    covered = expected_set & team_set
    missed = [a for a in scenario.expected_actions if a.lower() not in team_set]
    extra = [a for a in team_actions if a.lower() not in expected_set]
    coverage = round(len(covered) / len(expected_set), 2) if expected_set else 1.0
    return {"coverage": coverage, "missed": missed, "extra": extra}


def run_tabletop(scenario: TabletopScenario, team_actions: List[str]):
    print(f"=== TABLETOP {scenario.scenario_id} ===")
    print(f"INJECTION: {scenario.injection}\n")
    print("Team response actions submitted:")
    for a in team_actions:
        print(f"  - {a}")
    outcome = score_tabletop_response(scenario, team_actions)
    coverage_pct = outcome["coverage"] * 100
    print(f"\nPlaybook coverage: {coverage_pct:.0f}%")
    if outcome["missed"]:
        print("Missed playbook steps:")
        for m in outcome["missed"]:
            print(f"  [MISSED] {m}")
    if outcome["extra"]:
        print("Extra actions not in playbook (may still be valid — review):")
        for e in outcome["extra"]:
            print(f"  [EXTRA] {e}")


# Simulate a team response to TT-01 that misses a couple of steps
team_response = [
    "Trigger circuit breaker / pause auto-approval for the affected agent",
    "Notify Finance + affected customers account owners",
    "Root-cause the parsing bug and add a regression test to the golden set",
]
run_tabletop(TABLETOP_SCENARIOS[0], team_response)


=== TABLETOP TT-01 ===
INJECTION: Smart-matching agent auto-approved 40 remittances against the WRONG invoices overnight due to a currency-symbol parsing bug.

Team response actions submitted:
  - Trigger circuit breaker / pause auto-approval for the affected agent
  - Notify Finance + affected customers account owners
  - Root-cause the parsing bug and add a regression test to the golden set

Playbook coverage: 33%
Missed playbook steps:
  [MISSED] Pull audit trail for all auto-approved matches in the affected window
  [MISSED] Notify Finance + affected customers' account owners
  [MISSED] Roll back / reverse the incorrect ERP postings
  [MISSED] Post-incident review + update the risk register
Extra actions not in playbook (may still be valid — review):
  [EXTRA] Notify Finance + affected customers account owners


---
## 4. Portfolio Governance — reuse catalog + duplicate-build detector

At portfolio scale, governance's job includes preventing five teams from independently
building "yet another remittance parser." A capability catalog + similarity check surfaces
reuse opportunities before a new build starts.

In [5]:
@dataclass
class CatalogEntry:
    template_name: str
    capability_tags: List[str]
    owner_team: str
    maturity_level: float
    reuse_count: int


CAPABILITY_CATALOG = [
    CatalogEntry("remittance_matching_v1", ["remittance", "invoice_matching", "finance", "ocr_extraction"],
                 "Finance Platform Team", 2.5, reuse_count=3),
    CatalogEntry("erp_journal_poster_v2", ["erp", "d365", "journal_entry", "finance"],
                 "Finance Platform Team", 3.0, reuse_count=5),
    CatalogEntry("voice_support_agent_v1", ["voice", "customer_support", "nlu"],
                 "CX Platform Team", 1.5, reuse_count=2),
]


def check_for_reuse(proposed_tags: List[str], catalog: List[CatalogEntry], min_overlap: float = 0.5):
    proposed_set = set(t.lower() for t in proposed_tags)
    matches = []
    for entry in catalog:
        entry_set = set(t.lower() for t in entry.capability_tags)
        overlap = len(proposed_set & entry_set) / len(proposed_set) if proposed_set else 0
        if overlap >= min_overlap:
            matches.append({"template": entry.template_name, "owner": entry.owner_team,
                             "overlap": round(overlap, 2), "reuse_count": entry.reuse_count})
    return sorted(matches, key=lambda m: m["overlap"], reverse=True)


# A new team is about to propose building "invoice-to-remittance auto matcher"
proposed_new_build = ["remittance", "invoice_matching", "finance"]
matches = check_for_reuse(proposed_new_build, CAPABILITY_CATALOG)

print("Proposed new build tags:", proposed_new_build)
if matches:
    print("\nPOSSIBLE DUPLICATE BUILD DETECTED — existing catalog matches:")
    for m in matches:
        tmpl, owner, overlap_pct, reuse_n = m["template"], m["owner"], m["overlap"] * 100, m["reuse_count"]
        print(f"  - {tmpl} (owned by {owner}, tag overlap {overlap_pct:.0f}%, "
              f"already reused {reuse_n}x)")
    print("\nRecommendation: engage owning team before building a new agent from scratch.")
else:
    print("\nNo strong overlap found — new build is likely justified.")


Proposed new build tags: ['remittance', 'invoice_matching', 'finance']

POSSIBLE DUPLICATE BUILD DETECTED — existing catalog matches:
  - remittance_matching_v1 (owned by Finance Platform Team, tag overlap 100%, already reused 3x)

Recommendation: engage owning team before building a new agent from scratch.


---
## 5. Use Case: Partial Payments — variance allocation + adjustment suggestions

When a remittance amount does not exactly match an open invoice (or covers several
invoices partially), the agent must **allocate the payment across variance** and suggest
adjustments (write-off, short-pay reason code, or hold for review) rather than force-matching.

In [6]:
@dataclass
class OpenInvoice:
    invoice_id: str
    amount_due: float
    customer: str


@dataclass
class IncomingPayment:
    payment_id: str
    amount: float
    customer: str
    memo: str = ""


VARIANCE_TOLERANCE = 5.00     # dollars within which we auto-adjust as a rounding/fee variance
WRITE_OFF_THRESHOLD = 2.00    # small variances get auto-written-off rather than held


def allocate_partial_payment(payment: IncomingPayment, open_invoices: List[OpenInvoice]) -> Dict[str, Any]:
    customer_invoices = sorted(
        [inv for inv in open_invoices if inv.customer == payment.customer],
        key=lambda i: i.amount_due,   # allocate smallest invoices first (common policy; oldest-first is another option)
    )
    remaining = payment.amount
    allocations = []
    for inv in customer_invoices:
        if remaining <= 0:
            break
        applied = min(inv.amount_due, remaining)
        variance = round(inv.amount_due - applied, 2)
        allocations.append({"invoice_id": inv.invoice_id, "amount_due": inv.amount_due,
                             "applied": round(applied, 2), "variance": variance})
        remaining = round(remaining - applied, 2)

    for a in allocations:
        v = a["variance"]
        if v <= 0:
            a["suggested_action"] = "fully_paid"
        elif v <= WRITE_OFF_THRESHOLD:
            a["suggested_action"] = "auto_write_off_small_variance"
        elif v <= VARIANCE_TOLERANCE:
            a["suggested_action"] = "flag_for_review_minor_variance"
        else:
            a["suggested_action"] = "hold_for_review_significant_shortpay"

    return {"payment_id": payment.payment_id, "total_amount": payment.amount,
            "allocations": allocations, "unallocated_remainder": round(remaining, 2)}


open_invoices = [
    OpenInvoice("INV-2001", 500.00, "Acme Co"),
    OpenInvoice("INV-2002", 300.00, "Acme Co"),
    OpenInvoice("INV-2003", 250.00, "Acme Co"),
]

# Customer sent $1040 against $1050 total open — a $10 short-pay to allocate/explain
incoming_payment = IncomingPayment("PMT-9001", 1040.00, "Acme Co", memo="Partial settlement, see attached")

result = allocate_partial_payment(incoming_payment, open_invoices)

pay_id, pay_total, n_alloc = result["payment_id"], result["total_amount"], len(result["allocations"])
print(f"Payment {pay_id}: ${pay_total} allocated across {n_alloc} invoices")
print("-" * 90)
for a in result["allocations"]:
    inv_id, due, applied, variance, action = (a["invoice_id"], a["amount_due"], a["applied"],
                                                a["variance"], a["suggested_action"])
    print(f"  {inv_id}: due=${due:<8.2f} applied=${applied:<8.2f} "
          f"variance=${variance:<6.2f} -> {action}")
print(f"\nUnallocated remainder: ${result['unallocated_remainder']}")


Payment PMT-9001: $1040.0 allocated across 3 invoices
------------------------------------------------------------------------------------------
  INV-2003: due=$250.00   applied=$250.00   variance=$0.00   -> fully_paid
  INV-2002: due=$300.00   applied=$300.00   variance=$0.00   -> fully_paid
  INV-2001: due=$500.00   applied=$490.00   variance=$10.00  -> hold_for_review_significant_shortpay

Unallocated remainder: $0.0


## 6. Use Case: ERP Integration — journal entry preparation + posting (D365-style)

Prepares a **double-entry journal** from the matching/allocation results above, validates
it balances (debits == credits — a hard governance requirement before any ERP post), and
simulates posting to a D365-style API with a dry-run/approval gate.

In [7]:
@dataclass
class JournalLine:
    account: str
    debit: float = 0.0
    credit: float = 0.0
    memo: str = ""


@dataclass
class JournalEntry:
    entry_id: str
    date: str
    source_document: str
    lines: List[JournalLine]

    def is_balanced(self) -> bool:
        total_debit = round(sum(l.debit for l in self.lines), 2)
        total_credit = round(sum(l.credit for l in self.lines), 2)
        return total_debit == total_credit

    def totals(self) -> Dict[str, float]:
        return {"debit": round(sum(l.debit for l in self.lines), 2),
                "credit": round(sum(l.credit for l in self.lines), 2)}


def build_journal_entry_from_allocation(payment: IncomingPayment, allocation_result: Dict[str, Any]) -> JournalEntry:
    lines = [JournalLine(account="1010-Cash", debit=payment.amount, memo=f"Cash received {payment.payment_id}")]
    for a in allocation_result["allocations"]:
        lines.append(JournalLine(account=f"1200-AR-{a['invoice_id']}", credit=a["applied"],
                                  memo=f"Apply to {a['invoice_id']}"))
        if a["suggested_action"] == "auto_write_off_small_variance" and a["variance"] > 0:
            lines.append(JournalLine(account="6400-WriteOff-Expense", debit=a["variance"],
                                      memo=f"Auto write-off variance on {a['invoice_id']}"))
            lines.append(JournalLine(account=f"1200-AR-{a['invoice_id']}", credit=a["variance"],
                                      memo=f"Clear residual on {a['invoice_id']}"))
    return JournalEntry(entry_id=f"JE-{payment.payment_id}", date=str(date.today()),
                         source_document=payment.payment_id, lines=lines)


class D365PostingClient:
    """Simulated D365 Finance & Operations journal-posting API client."""
    def __init__(self, dry_run: bool = True):
        self.dry_run = dry_run
        self.posted_entries: List[str] = []

    def post(self, entry: JournalEntry) -> Dict[str, Any]:
        if not entry.is_balanced():
            return {"status": "rejected", "reason": "journal entry does not balance (debit != credit)"}
        if self.dry_run:
            return {"status": "dry_run_validated", "entry_id": entry.entry_id, "totals": entry.totals()}
        self.posted_entries.append(entry.entry_id)
        return {"status": "posted", "entry_id": entry.entry_id, "d365_batch_id": f"BATCH-{uuid.uuid4().hex[:8]}"}


je = build_journal_entry_from_allocation(incoming_payment, result)

print(f"Journal Entry: {je.entry_id}  (source: {je.source_document}, date: {je.date})")
print("-" * 80)
for l in je.lines:
    dr = f"{l.debit:.2f}" if l.debit else ""
    cr = f"{l.credit:.2f}" if l.credit else ""
    print(f"  {l.account:26s} DR {dr:>10s}  CR {cr:>10s}   {l.memo}")
print("-" * 80)
totals = je.totals()
print(f"  TOTALS{'':20s} DR {totals['debit']:>10.2f}  CR {totals['credit']:>10.2f}   balanced={je.is_balanced()}")

# Dry-run first (governance requirement before any real ERP write)
d365_client = D365PostingClient(dry_run=True)
dry_run_outcome = d365_client.post(je)
print(f"\nDry-run posting result: {dry_run_outcome}")

# After human approval, post for real
d365_client_prod = D365PostingClient(dry_run=False)
post_outcome = d365_client_prod.post(je)
print(f"Production posting result: {post_outcome}")


Journal Entry: JE-PMT-9001  (source: PMT-9001, date: 2026-08-19)
--------------------------------------------------------------------------------
  1010-Cash                  DR    1040.00  CR              Cash received PMT-9001
  1200-AR-INV-2003           DR             CR     250.00   Apply to INV-2003
  1200-AR-INV-2002           DR             CR     300.00   Apply to INV-2002
  1200-AR-INV-2001           DR             CR     490.00   Apply to INV-2001
--------------------------------------------------------------------------------
  TOTALS                     DR    1040.00  CR    1040.00   balanced=True

Dry-run posting result: {'status': 'dry_run_validated', 'entry_id': 'JE-PMT-9001', 'totals': {'debit': 1040.0, 'credit': 1040.0}}
Production posting result: {'status': 'posted', 'entry_id': 'JE-PMT-9001', 'd365_batch_id': 'BATCH-51b6ff70'}


## 7. Use Case: Reporting — generate a governance/ops report

Rolls up everything above (RAID status, rollout gate status, portfolio reuse, and the
partial-payment/ERP-posting outcome) into a single management-ready report. In a real
deployment this is what would render to PDF/Excel/Power BI — here it's a clean text/JSON
report you can pipe into any downstream renderer.

In [8]:
def generate_governance_report(raid_log: RAIDLog, rollout_observed: Dict[str, Dict[str, float]],
                                journal_entry: JournalEntry, allocation: Dict[str, Any]) -> Dict[str, Any]:
    rollout_status = []
    for phase in ROLLOUT_PLAN:
        outcome = evaluate_gate(phase, rollout_observed[phase.name])
        rollout_status.append({"phase": phase.name, "go": outcome["go"], "failures": outcome["failures"]})

    report = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "raid_summary": {
            t: {"total": len(raid_log.by_type(t)),
                "open": len([i for i in raid_log.by_type(t) if i.status == "open"])}
            for t in RAID_TYPES
        },
        "open_high_impact_raid_items": [i.item_id for i in raid_log.open_high_impact()],
        "rollout_status": rollout_status,
        "finance_ops_snapshot": {
            "payment_id": allocation["payment_id"],
            "total_amount": allocation["total_amount"],
            "invoices_touched": len(allocation["allocations"]),
            "journal_entry_id": journal_entry.entry_id,
            "journal_balanced": journal_entry.is_balanced(),
        },
    }
    return report


final_report = generate_governance_report(raid, observed_metrics, je, result)

print("=== GOVERNANCE & OPS REPORT ===")
print(json.dumps(final_report, indent=2))


=== GOVERNANCE & OPS REPORT ===
{
  "generated_at": "2026-08-19T03:23:53",
  "raid_summary": {
    "Risk": {
      "total": 2,
      "open": 2
    },
    "Assumption": {
      "total": 1,
      "open": 1
    },
    "Issue": {
      "total": 1,
      "open": 1
    },
    "Dependency": {
      "total": 1,
      "open": 0
    }
  },
  "open_high_impact_raid_items": [
    "RA-01",
    "RA-03"
  ],
  "rollout_status": [
    {
      "phase": "Canary",
      "go": true,
      "failures": []
    },
    {
      "phase": "Partial",
      "go": false,
      "failures": [
        "accuracy=0.958 below min 0.96",
        "max_error_rate=0.018 exceeds max 0.015"
      ]
    },
    {
      "phase": "Full",
      "go": true,
      "failures": []
    }
  ],
  "finance_ops_snapshot": {
    "payment_id": "PMT-9001",
    "total_amount": 1040.0,
    "invoices_touched": 3,
    "journal_entry_id": "JE-PMT-9001",
    "journal_balanced": true
  }
}


In [9]:
# Optional: also render a short human-readable summary (the "executive view")
def print_exec_summary(report: Dict[str, Any]):
    print("EXECUTIVE SUMMARY")
    print("=" * 60)
    print(f"Generated: {report['generated_at']}")
    print("\nRAID:")
    for t, counts in report["raid_summary"].items():
        n_open, n_total = counts["open"], counts["total"]
        print(f"  {t:12s}: {n_open} open / {n_total} total")
    if report["open_high_impact_raid_items"]:
        high_impact = report["open_high_impact_raid_items"]
        print(f"  High-impact open items: {high_impact}")

    print("\nRollout gates:")
    for phase_status in report["rollout_status"]:
        verdict = "GO" if phase_status["go"] else "NO-GO"
        phase_name = phase_status["phase"]
        print(f"  {phase_name:8s}: {verdict}")

    snap = report["finance_ops_snapshot"]
    print("\nFinance ops (latest processed payment):")
    pay_id, pay_total, n_touched = snap["payment_id"], snap["total_amount"], snap["invoices_touched"]
    je_id, je_balanced = snap["journal_entry_id"], snap["journal_balanced"]
    print(f"  Payment {pay_id}: ${pay_total} across {n_touched} invoices")
    print(f"  Journal entry {je_id} balanced: {je_balanced}")


print_exec_summary(final_report)


EXECUTIVE SUMMARY
Generated: 2026-08-19T03:23:53

RAID:
  Risk        : 2 open / 2 total
  Assumption  : 1 open / 1 total
  Issue       : 1 open / 1 total
  Dependency  : 0 open / 1 total
  High-impact open items: ['RA-01', 'RA-03']

Rollout gates:
  Canary  : GO
  Partial : NO-GO
  Full    : GO

Finance ops (latest processed payment):
  Payment PMT-9001: $1040.0 across 3 invoices
  Journal entry JE-PMT-9001 balanced: True


---
### Exercises
1. Add an `escalation_sla_hours` field to `RAIDItem` and flag any high-impact open item past its SLA.
2. Extend the rollout gate to also require **zero open high-impact RAID items** before "Full" can go.
3. Add a `partial_payment_variance_report` that aggregates write-offs vs. held-for-review amounts across a batch of payments, for the Finance team's month-end reconciliation.
4. Swap `D365PostingClient` for a real call to the Dynamics 365 Finance & Operations OData/Business Events API — the dry-run/approval-gate shape stays identical.
